# Pendolo Semplice - Soluzione Numerica

## Introduzione

Questo notebook simula il movimento di un **pendolo semplice senza attrito** usando l'**integrazione numerica**.

L'equazione differenziale non lineare che governa il moto è:

$$\frac{d^2\theta}{dt^2} + \frac{g}{L} \sin(\theta) = 0$$

dove:
- $\theta$ è l'angolo rispetto alla verticale (rad)
- $L$ è la lunghezza del pendolo (m)
- $g$ è l'accelerazione di gravità (m/s²)

### Approccio: Integrazione Numerica

Convertiamo l'equazione del secondo ordine in un sistema di due equazioni del primo ordine:

$$\frac{d\theta}{dt} = \omega$$
$$\frac{d\omega}{dt} = -\frac{g}{L}\sin(\theta)$$

Useremo `scipy.integrate.solve_ivp()` per risolvere numericamente questo sistema.

In [ ]:
# Importare le librerie necessarie
from pathlib import Path
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Configurazione di matplotlib per il notebook
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Parametri Fisici e Numerici

In [ ]:
# Parametri fisici
L = 1.0      # Lunghezza del pendolo (m)
g = 9.81     # Accelerazione di gravità (m/s²)

# Condizioni iniziali
theta0 = 0.3  # Angolo iniziale (rad)
omega0 = 0.0  # Velocità angolare iniziale (rad/s)

# Parametri di simulazione
t_max = 10.0    # Tempo massimo (s)
n_points = 1000 # Numero di punti temporali

print(f"Parametri della simulazione:")
print(f"  Lunghezza pendolo: L = {L} m")
print(f"  Gravità: g = {g} m/s²")
print(f"  Condizione iniziale: θ₀ = {theta0} rad ({np.degrees(theta0):.2f}°)")
print(f"  Velocità iniziale: ω₀ = {omega0} rad/s")
print(f"  Tempo simulato: 0 → {t_max} s")

## Funzione del Sistema Differenziale

Definiamo la funzione che restituisce le derivate del sistema:

$$\begin{cases}
\frac{d\theta}{dt} = \omega \\
\frac{d\omega}{dt} = -\frac{g}{L}\sin(\theta)
\end{cases}$$

In [ ]:
def pendolo_non_lineare(t, stato):
    """
    Sistema di equazioni differenziali per il pendolo semplice non lineare.
    
    Parametri:
        t: tempo (non usato, ma richiesto da solve_ivp)
        stato: vettore [θ, ω]
    
    Restituisce:
        [dθ/dt, dω/dt] = [ω, -(g/L)*sin(θ)]
    """
    theta, omega = stato
    dtheta_dt = omega
    domega_dt = -(g / L) * np.sin(theta)
    return [dtheta_dt, domega_dt]

# Test rapido della funzione
stato_test = [theta0, omega0]
derivate = pendolo_non_lineare(0, stato_test)
print(f"Derivate iniziali: dθ/dt = {derivate[0]:.4f}, dω/dt = {derivate[1]:.4f}")

## Integrazione Numerica

Usiamo `scipy.integrate.solve_ivp()` per risolvere il sistema di equazioni differenziali.

In [ ]:
# Vettore dei tempi in cui valutare la soluzione
t_eval = np.linspace(0.0, t_max, n_points)

# Stato iniziale: [θ(0), ω(0)]
stato_iniziale = [theta0, omega0]

# Risolvere il sistema di ODE
soluzione = solve_ivp(
    pendolo_non_lineare,
    t_span=(0.0, t_max),
    y0=stato_iniziale,
    t_eval=t_eval,
    rtol=1e-9,
    atol=1e-11,
    method='RK45'  # Runge-Kutta 4-5
)

if soluzione.success:
    print("✓ Integrazione completata con successo!")
    print(f"  Metodo: {soluzione.method}")
    print(f"  Numero di step: {soluzione.t_events}")
else:
    print(f"✗ Errore durante l'integrazione: {soluzione.message}")

# Estrarre le soluzioni
t = soluzione.t
theta = soluzione.y[0]
omega = soluzione.y[1]

# Calcolare l'accelerazione angolare dall'equazione del moto
alpha = -(g / L) * np.sin(theta)

print(f"\nRisultati:")
print(f"  θ_max = {theta.max():.4f} rad ({np.degrees(theta.max()):.2f}°)")
print(f"  θ_min = {theta.min():.4f} rad ({np.degrees(theta.min()):.2f}°)")
print(f"  ω_max = {omega.max():.4f} rad/s")
print(f"  ω_min = {omega.min():.4f} rad/s")

## Visualizzazione dei Risultati

Creiamo tre grafici che mostrano l'evoluzione temporale della posizione, velocità e accelerazione angolare.

In [ ]:
# Creare tre subplot per i tre grafici
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Grafico 1: Posizione angolare
axes[0].plot(t, theta, 'steelblue', linewidth=2, label='θ(t)')
axes[0].set_xlabel('Tempo t (s)', fontsize=11)
axes[0].set_ylabel('Posizione angolare θ (rad)', fontsize=11)
axes[0].set_title('Pendolo semplice: Posizione Angolare', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=10)

# Grafico 2: Velocità angolare
axes[1].plot(t, omega, 'darkorange', linewidth=2, label='ω(t)')
axes[1].set_xlabel('Tempo t (s)', fontsize=11)
axes[1].set_ylabel('Velocità angolare ω (rad/s)', fontsize=11)
axes[1].set_title('Pendolo semplice: Velocità Angolare', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=10)

# Grafico 3: Accelerazione angolare
axes[2].plot(t, alpha, 'crimson', linewidth=2, label='α(t)')
axes[2].set_xlabel('Tempo t (s)', fontsize=11)
axes[2].set_ylabel('Accelerazione angolare α (rad/s²)', fontsize=11)
axes[2].set_title('Pendolo semplice: Accelerazione Angolare', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Grafici completati!")